In [4]:
# Cell 1: Setup for TinyLlama LoRA training on local CAP JSONL files (CPU)

# Run once if needed:
# %pip install -q -U transformers datasets peft accelerate sentencepiece

import glob
import html
import os
import random
import re

import numpy as np
import torch
from datasets import DatasetDict, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, os.cpu_count() - 1))
print(f"Using device: {DEVICE}")
print(f"CPU threads: {torch.get_num_threads()}")

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
CAP_GLOB = "./cap_*.jsonl"
OUTPUT_DIR = "./tinyllama-cap-lora-cpu"
TOKENIZED_DIR = "./cap-tokenized-cache"

# CPU-friendly defaults (increase later if needed)
MAX_DOCS = 0   # set 0 to use all CAP docs
EVAL_RATIO = 0.02
MAX_LENGTH = 256
STRIDE = 64

MOJIBAKE_FIXES = {
    "â€™": "'",
    "â€œ": '"',
    "â€\x9d": '"',
    "â€“": "-",
    "â€”": "-",
    "Â§": "§",
    "Â": "",
}

def clean_text(text: str) -> str:
    text = html.unescape(text or "")
    for bad, good in MOJIBAKE_FIXES.items():
        text = text.replace(bad, good)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[\t ]+", " ", text)
    return text.strip()

Using device: cpu
CPU threads: 23


In [5]:
# Cell 2: Load CAP JSONL files and tokenize from the `text` field

cap_files = sorted(glob.glob(CAP_GLOB))
if not cap_files:
    raise FileNotFoundError(f"No files matched: {CAP_GLOB}")

print("CAP files found:")
for f in cap_files:
    print(" -", f)

raw_ds = load_dataset("json", data_files=cap_files, split="train")
raw_ds = raw_ds.filter(lambda x: isinstance(x.get("text"), str) and len(x["text"].strip()) > 0)

if MAX_DOCS > 0:
    raw_ds = raw_ds.shuffle(seed=SEED).select(range(min(MAX_DOCS, len(raw_ds))))

splits = raw_ds.train_test_split(test_size=EVAL_RATIO, seed=SEED)

train_docs = splits["train"].map(lambda b: {"text": [clean_text(t) for t in b["text"]]}, batched=True)
eval_docs = splits["test"].map(lambda b: {"text": [clean_text(t) for t in b["text"]]}, batched=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_batch(batch):
    tok = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
        padding=False,
    )
    tok["labels"] = tok["input_ids"].copy()
    return tok

# Important: when using return_overflowing_tokens=True, remove ALL original columns.
# Otherwise Arrow expects original batch length and throws a length mismatch error.
train_tokenized = train_docs.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_docs.column_names,
)
eval_tokenized = eval_docs.map(
    tokenize_batch,
    batched=True,
    remove_columns=eval_docs.column_names,
)

tokenized_ds = DatasetDict({"train": train_tokenized, "eval": eval_tokenized})
os.makedirs(TOKENIZED_DIR, exist_ok=True)
tokenized_ds.save_to_disk(TOKENIZED_DIR)

print(f"Train docs:   {len(train_docs)}")
print(f"Eval docs:    {len(eval_docs)}")
print(f"Train chunks: {len(train_tokenized)}")
print(f"Eval chunks:  {len(eval_tokenized)}")
print(f"Tokenized cache saved to: {TOKENIZED_DIR}")
print("Sample snippet:\n", train_docs[0]["text"][:500])

CAP files found:
 - .\cap_00000.jsonl
 - .\cap_00001.jsonl
 - .\cap_00002.jsonl
 - .\cap_00003.jsonl
 - .\cap_00004.jsonl


Saving the dataset (1/1 shards): 100%|██████████| 98493/98493 [00:00<00:00, 127187.19 examples/s]

Train docs:   196000
Eval docs:    4000
Train chunks: 4699483
Eval chunks:  98493
Tokenized cache saved to: ./cap-tokenized-cache
Sample snippet:
 In re Roberto Ortiz Gutiérrez, querellado.
 
 Número: CP-1998-7
 
 Resuelto: 18 de enero de 2001
 
 
 Carlos Lugo Fiol, Procurador General, e Ivonne Casanova Pe-losi, Procuradora General Auxiliar, querellantes; José F. Rodríguez Rivera, Comisionado Especial; Héctor M. Torres Rodríguez, abogado del querellado.
 per curiam:

El 15 de junio de 1998, el Procurador General de Puerto Rico presentó una querella ante nos contra el Ledo. Roberto Ortiz Gutiérrez mediante la cual le imputó los cargos sigui


In [ ]:
# Cell 3: Train TinyLlama + LoRA on CPU using CAP tokenized chunks

import inspect

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
    low_cpu_mem_usage=True,
)
model.to(DEVICE)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Build TrainingArguments in a version-safe way
all_args = {
    "output_dir": OUTPUT_DIR,
    "overwrite_output_dir": True,
    "num_train_epochs": 1,
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "evaluation_strategy": "steps",
    "eval_steps": 200,
    "logging_steps": 20,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "weight_decay": 0.01,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "fp16": False,
    "bf16": False,
    "dataloader_num_workers": 0,
    "report_to": "none",
    "optim": "adamw_torch",
}

supported = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
filtered_args = {k: v for k, v in all_args.items() if k in supported}

training_args = TrainingArguments(**filtered_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

train_result = trainer.train()
print(train_result)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6453.22it/s]


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'overwrite_output_dir'

In [ ]:
# Cell 4: Save LoRA adapter and run a quick legal inference sample

ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
FINAL_DIR = f"{OUTPUT_DIR}/final"

trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Optional full trainer save
trainer.save_model(FINAL_DIR)

print(f"LoRA adapter saved to: {ADAPTER_DIR}")

model.eval()
prompt = (
    "Summarize the legal holding and rationale from this case excerpt:\n\n"
    "The court reviewed whether the statutory damages cap violated equal protection, "
    "the remedies clause, and trial-by-jury guarantees under the state constitution."
)

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=140,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

result = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(result)

print("\nDone. Reload later with base TinyLlama + PeftModel.from_pretrained(adapter_path).")